# 面试问题：NLP 数据质量流水线应该检查什么，怎样避免清洗造成数据泄漏？

可以直接复述的回答是：原始文本要先做 Unicode 与空白规范化，再检查编码、语言、长度、标签、PII 和重复。每次拒绝或修正都应写事件账本，不能静默丢数据。去重不能只在 train/test 各自内部执行，否则全角、大小写或空白差异的同一文本仍会跨 split 泄漏。正确流程先生成规范化指纹或近重复组，再以组为单位切分。质量门禁还应保存规则版本、输入哈希和样本统计。下面用十二条客服语料手写清洗、规则、全局去重与 group split。

## 真实案例：客服意图训练集的十二条入库记录

记录包含文本、标签、来源和预分配 split；其中有全角重复、跨 split 重复、空文本、非法标签、英语样本和模拟邮箱。所有标识和邮箱均为教学构造，不是真实 PII。

In [1]:
import hashlib  # 导入稳定文本指纹函数
import re  # 导入空白、邮箱和语言规则
import unicodedata  # 导入 Unicode NFKC 规范化
records = [  # 定义十二条带质量问题的客服语料
    {"id": "DQ-01", "text": "退款  什么时候到账", "label": "refund", "source": "chat", "split": "train"},  # 多余空白但有效
    {"id": "DQ-02", "text": "包裹三天没有更新", "label": "shipping", "source": "chat", "split": "train"},  # 有效物流样本
    {"id": "DQ-03", "text": "设备无法开机", "label": "technical", "source": "ticket", "split": "train"},  # 有效技术样本
    {"id": "DQ-04", "text": "退款 什么时候到账", "label": "refund", "source": "ticket", "split": "test"},  # DQ-01 规范化后跨 split 重复
    {"id": "DQ-05", "text": "包裹三天没有更新", "label": "shipping", "source": "mail", "split": "test"},  # 完全跨 split 重复
    {"id": "DQ-06", "text": "ＡＰＰ 持续 闪退", "label": "technical", "source": "chat", "split": "train"},  # 含全角拉丁字符
    {"id": "DQ-07", "text": "APP 持续 闪退", "label": "technical", "source": "chat", "split": "test"},  # NFKC 后与 DQ-06 重复
    {"id": "DQ-08", "text": "", "label": "refund", "source": "chat", "split": "train"},  # 空文本应拒绝
    {"id": "DQ-09", "text": "会员如何取消", "label": "unknown", "source": "chat", "split": "train"},  # 非法标签应拒绝
    {"id": "DQ-10", "text": "please reset my password", "label": "technical", "source": "mail", "split": "train"},  # 非目标语言样本
    {"id": "DQ-11", "text": "联系 demo@example.test 处理退款", "label": "refund", "source": "ticket", "split": "train"},  # 含模拟邮箱需脱敏
    {"id": "DQ-12", "text": "怎么修改配送地址", "label": "shipping", "source": "chat", "split": "test"},  # 有效独立测试样本
]  # 结束十二条原始记录
allowed_labels = {"refund", "shipping", "technical"}  # 定义允许进入训练的标签集合
print("输入预览：id | split | source | label | raw_text")  # 输出原始语料表头
for record in records:  # 逐条展示十二条记录
    print(f"{record['id']} | {record['split']:5} | {record['source']:6} | {record['label']:9} | {record['text']!r}")  # 展示业务字段和质量问题
print("原始记录数：", len(records))  # 展示流水线输入规模

输入预览：id | split | source | label | raw_text
DQ-01 | train | chat   | refund    | '退款  什么时候到账'
DQ-02 | train | chat   | shipping  | '包裹三天没有更新'
DQ-03 | train | ticket | technical | '设备无法开机'
DQ-04 | test  | ticket | refund    | '退款 什么时候到账'
DQ-05 | test  | mail   | shipping  | '包裹三天没有更新'
DQ-06 | train | chat   | technical | 'ＡＰＰ 持续 闪退'
DQ-07 | test  | chat   | technical | 'APP 持续 闪退'
DQ-08 | train | chat   | refund    | ''
DQ-09 | train | chat   | unknown   | '会员如何取消'
DQ-10 | train | mail   | technical | 'please reset my password'
DQ-11 | train | ticket | refund    | '联系 demo@example.test 处理退款'
DQ-12 | test  | chat   | shipping  | '怎么修改配送地址'
原始记录数： 12


## Baseline / 基线：不做门禁直接按预分配 split 使用

基线接受全部十二条，只统计标签和 split；它会把空文本、非法标签与跨 split 重复都送入训练评估。

In [2]:
baseline_train = [record for record in records if record["split"] == "train"]  # 按原字段直接取训练集
baseline_test = [record for record in records if record["split"] == "test"]  # 按原字段直接取测试集
baseline_empty = sum(not record["text"].strip() for record in records)  # 统计空文本数量
baseline_bad_labels = sum(record["label"] not in allowed_labels for record in records)  # 统计非法标签数量
baseline_exact_leaks = []  # 收集原字符串完全相同的跨 split 泄漏
for train_record in baseline_train:  # 遍历基线训练记录
    for test_record in baseline_test:  # 遍历基线测试记录
        if train_record["text"] == test_record["text"]:  # 只用未经规范化的字符串比较
            baseline_exact_leaks.append((train_record["id"], test_record["id"]))  # 保存可见完全重复
print(f"baseline train={len(baseline_train)} test={len(baseline_test)} empty={baseline_empty} bad_label={baseline_bad_labels}")  # 输出基础质量统计
print("仅原字符串检测到的跨 split 重复：", baseline_exact_leaks)  # 展示基线漏掉全角和空白变体

baseline train=8 test=4 empty=1 bad_label=1
仅原字符串检测到的跨 split 重复： [('DQ-02', 'DQ-05')]


## 核心实现：规范化、PII 脱敏、规则门禁与事件账本

规则返回修正后的文本、状态和原因。英文比例只作为教学门禁；生产中应使用经过验证的语言识别器。

In [3]:
email_pattern = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")  # 定义模拟邮箱识别规则
def normalize_text(text):  # 执行确定性文本规范化
    normalized = unicodedata.normalize("NFKC", text)  # 把全角字符转换为兼容形式
    normalized = re.sub(r"\s+", " ", normalized).strip()  # 合并多余空白并去除首尾空格
    return normalized  # 返回规范化文本
def chinese_ratio(text):  # 粗略计算中文字符占非空白字符比例
    characters = [character for character in text if not character.isspace()]  # 去掉空白后统计语言字符
    chinese = sum("一" <= character <= "鿿" for character in characters)  # 统计基本中文字符数
    return chinese / len(characters) if characters else 0.0  # 返回比例并处理空文本
def inspect_record(record):  # 对一条记录执行修正与质量门禁
    normalized = normalize_text(record["text"])  # 先统一 Unicode 和空白
    redacted = email_pattern.sub("<EMAIL>", normalized)  # 将模拟邮箱替换为占位符
    reasons = []  # 收集拒绝或修正原因
    if normalized != record["text"]:  # 检查文本是否经过规范化修正
        reasons.append("normalized")  # 记录规范化事件
    if redacted != normalized:  # 检查是否发生 PII 脱敏
        reasons.append("email_redacted")  # 记录邮箱脱敏事件
    if not redacted:  # 检查清洗后空文本
        reasons.append("empty")  # 记录空文本拒绝原因
    if record["label"] not in allowed_labels:  # 检查标签字典约束
        reasons.append("invalid_label")  # 记录非法标签拒绝原因
    if redacted and chinese_ratio(redacted.replace("<EMAIL>", "")) < 0.30:  # 检查目标语种最低比例
        reasons.append("language_mismatch")  # 记录非目标语言拒绝原因
    rejected = any(reason in {"empty", "invalid_label", "language_mismatch"} for reason in reasons)  # 汇总硬拒绝状态
    fingerprint = hashlib.sha1(redacted.lower().encode("utf-8")).hexdigest()[:12]  # 计算规范化小写文本稳定指纹
    return {**record, "clean_text": redacted, "fingerprint": fingerprint, "status": "reject" if rejected else "accept", "reasons": reasons}  # 返回完整审计记录
audited = [inspect_record(record) for record in records]  # 对十二条输入逐条执行质量门禁
print("id | status | clean_text | fingerprint | reasons")  # 输出质量事件账本表头
for row in audited:  # 遍历十二条审计结果
    print(f"{row['id']} | {row['status']:6} | {row['clean_text']!r:28} | {row['fingerprint']} | {row['reasons']}")  # 展示每次修正和拒绝依据
accepted = [row for row in audited if row["status"] == "accept"]  # 形成通过硬质量门禁的候选集
print(f"门禁后 accepted={len(accepted)} rejected={len(audited) - len(accepted)}")  # 汇总规则执行结果

id | status | clean_text | fingerprint | reasons
DQ-01 | accept | '退款 什么时候到账'                  | 75e4090d23b2 | ['normalized']
DQ-02 | accept | '包裹三天没有更新'                   | 0842c0eee20a | []
DQ-03 | accept | '设备无法开机'                     | ce5496536dea | []
DQ-04 | accept | '退款 什么时候到账'                  | 75e4090d23b2 | []
DQ-05 | accept | '包裹三天没有更新'                   | 0842c0eee20a | []
DQ-06 | accept | 'APP 持续 闪退'                  | 56c63808420a | ['normalized']
DQ-07 | accept | 'APP 持续 闪退'                  | 56c63808420a | []
DQ-08 | reject | ''                           | da39a3ee5e6b | ['empty']
DQ-09 | reject | '会员如何取消'                     | fb2447ef24b9 | ['invalid_label']
DQ-10 | reject | 'please reset my password'   | 7f7affd0fb54 | ['language_mismatch']
DQ-11 | accept | '联系 <EMAIL> 处理退款'            | dad0b4085cc5 | ['email_redacted']
DQ-12 | accept | '怎么修改配送地址'                   | 67d7a6387085 | []
门禁后 accepted=9 rejected=3


## 全局去重与组级切分

相同 fingerprint 的样本先组成一组，再按组 ID 稳定切分。组内保留第一条作为训练语料，其他记录写入 duplicate 账本。

In [4]:
groups = {}  # 建立规范化指纹到记录列表映射
for row in accepted:  # 遍历通过门禁的记录
    groups.setdefault(row["fingerprint"], []).append(row)  # 把规范化重复放入同一组
deduplicated = []  # 收集每组保留的代表记录
duplicate_ledger = []  # 收集被折叠的重复记录
for fingerprint, members in groups.items():  # 遍历全部规范化文本组
    representative = sorted(members, key=lambda row: row["id"])[0]  # 用最小记录 ID 稳定选择代表
    group_split = "test" if int(fingerprint[-2:], 16) % 4 == 0 else "train"  # 按指纹而非单记录稳定切分
    deduplicated.append({**representative, "group_split": group_split})  # 保存代表记录和组级 split
    for member in members:  # 遍历当前组所有原始记录
        if member["id"] != representative["id"]:  # 检查是否为重复副本
            duplicate_ledger.append((member["id"], representative["id"], fingerprint))  # 记录副本到代表映射
group_train = [row for row in deduplicated if row["group_split"] == "train"]  # 形成无重复训练集
group_test = [row for row in deduplicated if row["group_split"] == "test"]  # 形成无重复测试集
train_fingerprints = {row["fingerprint"] for row in group_train}  # 获取训练指纹集合
test_fingerprints = {row["fingerprint"] for row in group_test}  # 获取测试指纹集合
print("duplicate | kept_as | fingerprint")  # 输出全局去重账本表头
for item in duplicate_ledger:  # 遍历规范化重复副本
    print(item)  # 展示空白和全角变体被合并
print("组级 train：", [(row["id"], row["clean_text"]) for row in group_train])  # 展示最终训练记录
print("组级 test：", [(row["id"], row["clean_text"]) for row in group_test])  # 展示最终测试记录
print("跨 split 指纹交集：", train_fingerprints & test_fingerprints)  # 验证切分后不存在同组泄漏

duplicate | kept_as | fingerprint
('DQ-04', 'DQ-01', '75e4090d23b2')
('DQ-05', 'DQ-02', '0842c0eee20a')
('DQ-07', 'DQ-06', '56c63808420a')
组级 train： [('DQ-01', '退款 什么时候到账'), ('DQ-02', '包裹三天没有更新'), ('DQ-03', '设备无法开机'), ('DQ-06', 'APP 持续 闪退'), ('DQ-11', '联系 <EMAIL> 处理退款'), ('DQ-12', '怎么修改配送地址')]
组级 test： []
跨 split 指纹交集： set()


## 失败案例与修正：只在各 split 内去重

错误方案先按原 split 分开，再各自去重，DQ-01/DQ-04、DQ-06/DQ-07 仍跨集合出现。修正方案在切分之前做全局 fingerprint grouping。

In [5]:
naive_train_fingerprints = {row["fingerprint"] for row in accepted if row["split"] == "train"}  # 计算错误训练 split 规范化指纹
naive_test_fingerprints = {row["fingerprint"] for row in accepted if row["split"] == "test"}  # 计算错误测试 split 规范化指纹
naive_leak_fingerprints = naive_train_fingerprints & naive_test_fingerprints  # 查找只做 split 内去重后的泄漏
naive_leak_pairs = []  # 收集泄漏指纹对应的可读记录对
for fingerprint in sorted(naive_leak_fingerprints):  # 遍历每个跨 split 泄漏组
    train_ids = [row["id"] for row in accepted if row["split"] == "train" and row["fingerprint"] == fingerprint]  # 查找训练侧记录
    test_ids = [row["id"] for row in accepted if row["split"] == "test" and row["fingerprint"] == fingerprint]  # 查找测试侧记录
    naive_leak_pairs.append((train_ids, test_ids, fingerprint))  # 保存可审计泄漏对
quality_summary = {"input": len(records), "accepted_before_dedup": len(accepted), "deduplicated": len(deduplicated), "rejected": len(records) - len(accepted), "duplicates": len(duplicate_ledger)}  # 形成最终质量统计
print("错误 split 内去重仍泄漏：", naive_leak_pairs)  # 展示真实跨集合污染
print("修正后跨 split 交集：", train_fingerprints & test_fingerprints)  # 展示组级切分消除泄漏
print("质量摘要：", quality_summary)  # 输出可供监控的计数器

错误 split 内去重仍泄漏： [(['DQ-02'], ['DQ-05'], '0842c0eee20a'), (['DQ-06'], ['DQ-07'], '56c63808420a'), (['DQ-01'], ['DQ-04'], '75e4090d23b2')]
修正后跨 split 交集： set()
质量摘要： {'input': 12, 'accepted_before_dedup': 9, 'deduplicated': 6, 'rejected': 3, 'duplicates': 3}


## 结果解读

NFKC 和空白规范化让肉眼不同的文本落入同一指纹组；事件账本同时保留脱敏、拒绝和重复原因。先切分再去重无法消除跨集合泄漏，正确顺序是规范化、全局分组、组级切分。

## 生产边界

教学规则只有十二条记录和简单语言比例。生产流水线还需编码失败隔离、近重复 MinHash、毒性/许可证、标签一致性、数据漂移与人工抽检，并以不可变快照保存规则版本。PII 检测不能只靠一个正则；删除请求还要沿 lineage 清理训练快照和派生特征。

## 最小回归测试

In [6]:
assert len(records) >= 6  # 保证案例包含多个真实语义记录
assert baseline_empty == 1 and baseline_bad_labels == 1  # 保证原始基线含可复现质量问题
assert len(accepted) < len(records) and len(deduplicated) < len(accepted)  # 保证门禁和全局去重均实际生效
assert len(duplicate_ledger) >= 3  # 保证空白、完全重复和全角变体均被折叠
assert len(naive_leak_fingerprints) >= 2  # 保证 split 内去重泄漏失败真实复现
assert train_fingerprints.isdisjoint(test_fingerprints)  # 保证组级切分消除规范化重复泄漏
assert any("email_redacted" in row["reasons"] for row in audited)  # 保证模拟 PII 修正写入审计账本